# XGBoost Distributed Training with GPU Scaling

Snowflake ML's `XGBEstimator` automatically distributes XGBoost training across all available GPUs — zero cluster config required.

**Scenario:** 200 products × 50 stores × 365 days of sales (~2.5M rows). We predict optimal selling prices using gradient-boosted trees.

**Flow:** Feature Store → GPU-Scaled XGBoost → Model Registry

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb


from snowflake.ml.data.data_connector import DataConnector
from snowflake.ml.modeling.preprocessing import LabelEncoder
from snowflake.ml.modeling.pipeline import Pipeline
from snowflake.snowpark.context import get_active_session

import warnings
warnings.filterwarnings('ignore')

session = get_active_session()

DB = "DYNAMIC_PRICING"
SCHEMA = "PRICING_MODEL"

session.use_schema(f"{DB}.{SCHEMA}")

print(f"Database: {DB}")
print(f"Schema: {SCHEMA}")

In [ ]:
from snowflake.ml.runtime_cluster import scale_cluster
# Run this to scale the underlying compute cluster to 3 nodes.
scale_cluster(3)

## 2. Feature Store — Retrieve Features

We connect to the **existing online feature store** that was set up for the Two-Tower pricing model. The feature store already has `PRODUCT_FEATURES` and `STORE_FEATURES` registered with online serving enabled.

In [ ]:
from snowflake.ml.feature_store import FeatureStore, CreationMode

FS_SCHEMA = "FS_SCHEMA"

fs = FeatureStore(
    session=session,
    database=DB,
    name=FS_SCHEMA,
    default_warehouse=session.get_current_warehouse(),
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

product_fv = fs.get_feature_view("PRODUCT_FEATURES", "V1")
store_fv = fs.get_feature_view("STORE_FEATURES", "V1")

print(f"Product Features: {product_fv.name} {product_fv.version}")
print(f"Store Features: {store_fv.name} {store_fv.version}")

## 3. Prepare Training Data

We build a **spine** of product-store pairs with their average selling price as the target, then join features from the feature store to create the full training dataset.

In [ ]:
spine_df = session.sql(f"""
    SELECT PRODUCT_ID, STORE_ID, ROUND(AVG(SELLING_PRICE), 2) AS TARGET_PRICE
    FROM {DB}.{SCHEMA}.FACT_SALES_DAILY
    GROUP BY PRODUCT_ID, STORE_ID
""")
print(f"Spine records (product-store pairs): {spine_df.count()}")

training_dataset = fs.generate_dataset(
    name="XGBOOST_TRAINING_DATA",
    spine_df=spine_df,
    features=[product_fv, store_fv],
    output_type="dataset",
)

training_df = training_dataset.read.to_snowpark_dataframe()
print(f"Training samples from Feature Store: {training_df.count()}")
training_df.show(5)

### Preprocessing

Label-encode categorical features. XGBoost is tree-based, so numeric features don't need scaling.

In [ ]:
PRODUCT_CAT_COLS = ["CATEGORY", "SUBCATEGORY", "BRAND"]
PRODUCT_NUM_COLS = ["COST", "BASE_PRICE", "WEIGHT"]
STORE_CAT_COLS = ["REGION", "CITY", "STORE_TYPE"]
STORE_NUM_COLS = ["LAT", "LON"]

INPUT_COLS = PRODUCT_CAT_COLS + PRODUCT_NUM_COLS + STORE_CAT_COLS + STORE_NUM_COLS
LABEL_COL = "TARGET_PRICE"

pipeline_steps = []
for col in PRODUCT_CAT_COLS + STORE_CAT_COLS:
    pipeline_steps.append((f"LE_{col}", LabelEncoder(input_cols=[col], output_cols=[col])))

pipeline = Pipeline(steps=pipeline_steps)
processed_df = pipeline.fit(training_df).transform(training_df)

print(f"Processed: {processed_df.count()} rows, {len(INPUT_COLS)} features")
processed_df.show(5)

In [ ]:
train_df, eval_df = processed_df.random_split([0.8, 0.2], seed=42)

train_connector = DataConnector.from_dataframe(train_df)
eval_connector = DataConnector.from_dataframe(eval_df)

print(f"Train: {train_df.count()} | Eval: {eval_df.count()}")

## 4. Distributed XGBoost Training (GPU Scaled)

`XGBEstimator` automatically discovers all available GPUs and distributes training across them — no manual cluster setup needed.

In [ ]:
import time
from snowflake.ml.modeling.distributors.xgboost import XGBEstimator, XGBScalingConfig

estimator = XGBEstimator(
    n_estimators=10,
    params={
        "tree_method": "hist",
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "max_depth": 6,
        "learning_rate": 0.1,
    },
    scaling_config=XGBScalingConfig(use_gpu=True),
)

start = time.time()

booster = estimator.fit(
    dataset=train_connector,
    input_cols=INPUT_COLS,
    label_col=LABEL_COL,
    eval_set=eval_connector,
    verbose_eval=50,
)

print(f"\nTraining complete in {time.time() - start:.1f}s")

## 5. Model Evaluation & Feature Importance

In [ ]:
eval_pd = eval_connector.to_pandas()
X_eval = eval_pd[INPUT_COLS]
y_eval = eval_pd[LABEL_COL]

deval = xgb.DMatrix(X_eval, label=y_eval)
preds = booster.predict(deval)

rmse = np.sqrt(np.mean((preds - y_eval.values) ** 2))
mae = np.mean(np.abs(preds - y_eval.values))
r2 = 1 - np.sum((y_eval.values - preds) ** 2) / np.sum((y_eval.values - y_eval.mean()) ** 2)

print(f"RMSE: ${rmse:.2f}  |  MAE: ${mae:.2f}  |  R²: {r2:.4f}")

importance = booster.get_score(importance_type='gain')
importance_sorted = sorted(importance.items(), key=lambda x: x[1], reverse=True)

print("\nFeature Importance (gain):")
for feat, gain in importance_sorted:
    bar = "█" * int(gain / max(importance.values()) * 30)
    print(f"  {feat:<15s} {gain:>10.1f}  {bar}")

In [ ]:
pd.DataFrame({
    "ACTUAL": y_eval.values[:10],
    "PREDICTED": np.round(preds[:10], 2),
    "ERROR": np.round(preds[:10] - y_eval.values[:10], 2),
})

In [ ]:
eval_df.write.mode("overwrite").save_as_table(f"{DB}.{SCHEMA}.XGB_EVAL_DATA")
print(f"Saved eval data for SQL inference: {DB}.{SCHEMA}.XGB_EVAL_DATA")

## 5b. SHAP Explainability

SHAP (SHapley Additive exPlanations) assigns each feature an importance value for every prediction. Unlike global feature importance (gain), SHAP values show **how much each feature pushed a specific prediction** above or below the average — giving per-prediction transparency.

- **Bar plot**: Mean absolute SHAP value per feature — the global ranking of feature impact.
- **Beeswarm plot**: Each dot is one prediction. Position on the x-axis shows the SHAP value (impact on output), color shows the feature's actual value (red = high, blue = low). This reveals **directional relationships** — e.g., higher `COST` pushes predicted price up.

In [ ]:
import shap
import xgboost as xgb

# Use XGBoost's built-in predict(pred_contribs=True) to compute SHAP values
deval_shap = xgb.DMatrix(X_eval)
shap_values = booster.predict(deval_shap, pred_contribs=True)

# Last column is the base value (bias); separate it out
base_value = shap_values[0, -1]
shap_values = shap_values[:, :-1]

print(f"SHAP values computed for {shap_values.shape[0]} samples x {shap_values.shape[1]} features")
print(f"Expected base value (avg prediction): ${base_value:.2f}")

In [ ]:
shap.summary_plot(shap_values, X_eval, plot_type="bar", show=True)
shap.summary_plot(shap_values, X_eval, show=True)

## 6. Model Registry

Register the booster directly — XGBoost is a built-in model type, so no custom wrapper is needed. To include the preprocessor pipeline in the model directly for inference, you would need to utilize a [Custom Model class](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/bring-your-own-model-types). 

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name=DB, schema_name=SCHEMA)

background_data = X_eval.sample(n=min(100, len(X_eval)), random_state=42)

mv = reg.log_model(
    booster,
    model_name="XGBOOST_PRICING",
    version_name="V1",
    sample_input_data=background_data,
    conda_dependencies=["snowflake-ml-python"],
    options={"enable_explainability": True},
    comment=f"XGBoost pricing model with Shapley explainability | RMSE: ${rmse:.2f} | R²: {r2:.4f}",
    target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"]
)

print(f"Registered: {mv.model_name} {mv.version_name}")
print(mv.show_functions())

In [ ]:
explain_input = X_eval.head(5)

predictions = mv.run(explain_input, function_name="predict")
print("Predictions:")
print(predictions)

explanations = mv.run(explain_input, function_name="explain")
print("\nShapley explanations (feature contributions to each prediction):")
print(explanations)

### SQL Inference Example

Once registered, the model is available for **native SQL inference** — no Python runtime needed:

```sql
SELECT
    PRODUCT_ID, STORE_ID,
    MODEL(DYNAMIC_PRICING.PRICING_MODEL.XGBOOST_PRICING, V1)!PREDICT(
        CATEGORY, SUBCATEGORY, BRAND, COST, BASE_PRICE, WEIGHT,
        REGION, CITY, STORE_TYPE, LAT, LON
    ):output_feature_0::FLOAT AS PREDICTED_PRICE
FROM DYNAMIC_PRICING.PRICING_MODEL.<TABLE WITH INPUT DATA>
LIMIT 10;
```